# GraphCypherQAChain으로 자연어 질문


```text
자연어 질문
  → LLM이 Cypher 생성
  → Neo4j 조회
  → LLM이 자연어 답변 생성
```

## 환경

### 1) Neo4j

Neo4j Desktop DB 실행

- Neo4j Browser: `http://localhost:7474`
- Bolt: `bolt://localhost:7687`
- 사용자: `neo4j`
- 비밀번호: `graphragpassword`

### 2) `.env`

```env
OPENAI_API_KEY=your_openai_api_key
OPENAI_MODEL=gpt-5.5

NEO4J_URI=bolt://localhost:7687
NEO4J_USERNAME=neo4j
NEO4J_PASSWORD=graphragpassword
NEO4J_DATABASE=neo4j
```

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

URI = os.getenv("NEO4J_URI")
USERNAME = os.getenv("NEO4J_USERNAME")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")
OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5.5")

print("URI:", URI or "(미설정)")
print("DATABASE:", DATABASE)

URI: bolt://localhost:7687
DATABASE: neo4j


## 1. 질문

고객에서 주문을 거쳐 택배사까지 두 관계를 연결해야 합니다.

In [2]:
# 자연어 질문 목록
questions = [
    "고객 A의 주문을 배송하는 택배사는 어디인가요?",
]

for question in questions:
    print(question)

고객 A의 주문을 배송하는 택배사는 어디인가요?


## 2. 사람이 작성한 기준 Cypher

LLM이 생성한 Cypher를 비교할 기준입니다.

In [3]:
REFERENCE_CYPHER = """
MATCH (customer:Customer {id: "고객 A"})
      -[:PLACED]->
      (order:Order)
      -[:SHIPPED_BY]->
      (courier:Courier)
RETURN courier.name AS courier
""".strip()

print(REFERENCE_CYPHER)

MATCH (customer:Customer {id: "고객 A"})
      -[:PLACED]->
      (order:Order)
      -[:SHIPPED_BY]->
      (courier:Courier)
RETURN courier.name AS courier


## 3. 예상 결과

In [4]:
expected_rows = [{"courier": "빠른택배"}]
expected_answer = "고객 A의 주문을 배송하는 택배사는 빠른택배입니다."

print("조회 결과:", expected_rows)
print("답변:", expected_answer)

조회 결과: [{'courier': '빠른택배'}]
답변: 고객 A의 주문을 배송하는 택배사는 빠른택배입니다.


## 4. GraphCypherQAChain


In [6]:
from langchain_openai import ChatOpenAI
from langchain_neo4j import Neo4jGraph, GraphCypherQAChain
from langchain_core.prompts import PromptTemplate

# Neo4j에 연결하고 현재 그래프 스키마를 읽는다.
graph = Neo4jGraph(
    url=URI,
    username=USERNAME,
    password=PASSWORD,
    database=DATABASE
)
# 현재 라벨, 관계 타입, 속성을 읽어 LLM 프롬프트의 {schema}에 제공함
graph.refresh_schema()

# 자연어 질문을 Cypher로 변환하고 조회 결과를 답변으로 만드는 LLM이다.
llm = ChatOpenAI(
    model=OPENAI_MODEL,
    temperature=0    
)

# 스키마와 질문을 받아 조회 전용 Cypher를 생성하게 하는 프롬프트 템플릿
cypher_generation_template = """
온라인 쇼핑몰 Neo4j 그래프를 조회하는 Cypher를 작성하세요.

규칙:
- 아래 스키마에 있는 라벨, 관계, 속성만 사용하세요.
- 고객, 주문, 상품, 물류센터, 택배사, 쿠폰의 식별자는 `name`이 아니라 `id`로 조회하세요.
- 사용자는 `ID`라는 단어 없이 일상적인 문장으로 질문할 수 있습니다.
- 질문에 나온 엔티티 표현을 접두어까지 포함한 전체 문자열로 인식하세요.
- `고객 A`는 Customer의 id `고객 A`, `주문 O-1001`은 Order의 id `주문 O-1001`입니다.
- `여름할인 쿠폰`은 Coupon의 id `여름할인 쿠폰`입니다.
- `"고객 A"`를 `"A"`로, `"주문 O-1001"`을 `"O-1001"`로 축약하면 안 됩니다.
- `출고 물류센터`는 `FULFILLED_BY` 관계를 사용하세요.
- 설명이나 Markdown 없이 Cypher만 반환하세요.

예시 1 질문:
고객 A의 주문을 배송하는 택배사는 어디인가요?

예시 1 Cypher:
MATCH (customer:Customer {{id: "고객 A"}})
      -[:PLACED]->(:Order)
      -[:SHIPPED_BY]->(courier:Courier)
RETURN DISTINCT courier.name AS courier

예시 2 질문:
주문 O-1001에는 어떤 상품이 들어 있나요?

예시 2 Cypher:
MATCH (order:Order {{id: "주문 O-1001"}})
      -[:CONTAINS]->(product:Product)
RETURN DISTINCT product.name AS product

예시 3 질문:
주문 O-1001은 어느 물류센터에서 출고되나요?

예시 3 Cypher:
MATCH (order:Order {{id: "주문 O-1001"}})
      -[:FULFILLED_BY]->(warehouse:Warehouse)
RETURN DISTINCT warehouse.name AS warehouse

예시 4 질문:
여름할인 쿠폰이 적용된 주문을 알려주세요.

예시 4 Cypher:
MATCH (order:Order)
      -[:USES_COUPON]->(coupon:Coupon {{id: "여름할인 쿠폰"}})
RETURN DISTINCT order.id AS order

스키마:
{schema}

질문:
{question}

Cypher:
""".strip()


cypher_prompt = PromptTemplate(
    input_variables=["schema", "question"],
    template=cypher_generation_template
)

# 체인 실행 흐름 : 질문 -> Cypher  생성 -> Neo4j 실행 -> 조회 결과의 자연어 요약
chain = GraphCypherQAChain.from_llm(
    llm=llm,
    graph=graph,
    cypher_prompt=cypher_prompt,
    verbose=True,
    validate_cypher=True,      # 생성 쿼리의 관계 방향 등 기본 구조 실행 전 검사
    # 생성 Cypher 실행 명시적으로 허용한다. 
    # 즉, 데이터 삭제, 수정이나 민감한 정보 조회가 발생할 수 있음
    allow_dangerous_requests=True
)

print("Graph schema:")
print(graph.schema)

for question in questions:
    print("\n질문:", question)
    result = chain.invoke({"query": question})
    print("답변:", result["result"])

Graph schema:
Node properties:
Customer {id: STRING, name: STRING, type: STRING}
Order {id: STRING, name: STRING, type: STRING}
Product {id: STRING, name: STRING, type: STRING}
Warehouse {id: STRING, name: STRING, type: STRING}
Courier {id: STRING, name: STRING, type: STRING}
Coupon {id: STRING, name: STRING, type: STRING}
Relationship properties:

The relationships:
(:Customer)-[:PLACED]->(:Order)
(:Order)-[:CONTAINS]->(:Product)
(:Order)-[:FULFILLED_BY]->(:Warehouse)
(:Order)-[:SHIPPED_BY]->(:Courier)
(:Order)-[:USES_COUPON]->(:Coupon)

질문: 고객 A의 주문을 배송하는 택배사는 어디인가요?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (customer:Customer {id: "고객 A"})
      -[:PLACED]->(:Order)
      -[:SHIPPED_BY]->(courier:Courier)
RETURN DISTINCT courier.name AS courier
Full Context:
[{'courier': '빠른택배'}]

> Finished chain.
답변: 고객 A의 주문을 배송하는 택배사는 빠른택배입니다.


## 5. 생성 Cypher 관찰 포인트

다음을 확인합니다.

1. `Customer`, `Order`, `Courier` 라벨을 사용했는가
2. `PLACED`, `SHIPPED_BY` 관계 방향이 맞는가
3. `id = "고객 A"` 조건이 있는가
4. 조회 결과가 필요 이상으로 크지 않은가

## 6. 추가 질문

In [ ]:
practice_questions = [
    "주문 O-1001에는 어떤 상품이 들어 있나요?",
    "주문 O-1001은 어느 물류센터에서 출고되나요?",
    "여름할인 쿠폰이 적용된 주문을 알려주세요.",
]

practice_results = []

for index, question in enumerate(practice_questions, start=1):
    print(f"\n[{index}] 질문: {question}")
    result = chain.invoke({"query": question})
    answer = result["result"]
    practice_results.append(
        {
            "question": question,
            "answer": answer,
        }
    )
    print("답변:", answer)



[1] 질문: 주문 O-1001에는 어떤 상품이 들어 있나요?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (order:Order {id: "주문 O-1001"})
      -[:CONTAINS]->(product:Product)
RETURN DISTINCT product.name AS product
Full Context:
[{'product': '무선 이어폰'}, {'product': '노트북 파우치'}]

> Finished chain.
답변: 주문 O-1001에는 무선 이어폰, 노트북 파우치가 들어 있습니다.

[2] 질문: 주문 O-1001은 어느 물류센터에서 출고되나요?


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (order:Order {id: "주문 O-1001"})
      -[:FULFILLED_BY]->(warehouse:Warehouse)
RETURN DISTINCT warehouse.name AS warehouse
Full Context:
[{'warehouse': '이천 물류센터'}]

> Finished chain.
답변: 주문 O-1001은 이천 물류센터에서 출고됩니다.

[3] 질문: 여름할인 쿠폰이 적용된 주문을 알려주세요.


> Entering new GraphCypherQAChain chain...
Generated Cypher:
MATCH (order:Order)
      -[:USES_COUPON]->(coupon:Coupon {id: "여름할인 쿠폰"})
RETURN DISTINCT order.id AS order
Full Context:
[{'order': '주문 O-1001'}]

> Finished chain.
답변: 여름할인 쿠폰이 적용된 주문은 주문 O-1001입니다.
